In [21]:
import math

from pyod.models import lof
from scipy.io import arff
from operator import index

import numpy as np
from sklearn.metrics import roc_auc_score
from sklearn.neighbors import NearestNeighbors, KernelDensity
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import scipy.stats
from scipy.stats import expon, skew, norm,gamma, anderson,goodness_of_fit, monte_carlo_test, probplot, skewnorm
from scipy import integrate
from sklearn.metrics import auc
import seaborn as sns
import math

from statsmodels.sandbox.distributions.gof_new import kstest

plt.rcParams['figure.figsize'] = [15, 7]
import warnings
from scipy import stats
from pyod.models import abod, knn, lof, cof, kde, sos, sod
warnings.filterwarnings('ignore')

In [22]:
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import roc_auc_score
from scipy.stats import norm, gaussian_kde
from itertools import combinations

def simplified_lof(X, k):
    nbrs = NearestNeighbors(n_neighbors=k+1, p=1).fit(X)
    distances, _ = nbrs.kneighbors(X)
    return distances[:, 1:].mean(axis=1)

# LDOF
def ldof(X, k):
    n = X.shape[0]
    nbrs = NearestNeighbors(n_neighbors=k+1, p=1).fit(X)
    distances, indices = nbrs.kneighbors(X)
    distances, indices = distances[:, 1:], indices[:, 1:]

    scores = np.zeros(n)
    for i in range(n):
        xi = X[i]
        neighbors = X[indices[i]]
        d_avg = np.mean(np.linalg.norm(neighbors - xi, axis=1, ord=1))
        d_pairwise = np.mean([np.linalg.norm(neighbors[a] - neighbors[b], ord=1) for a, b in combinations(range(k), 2)])
        scores[i] = d_avg / d_pairwise if d_pairwise != 0 else 0
    return scores

# ODIN
def odin(X, k):
    nbrs = NearestNeighbors(n_neighbors=k+1, p=1).fit(X)
    distances, _ = nbrs.kneighbors(X)
    return distances[:, -1]

# KDEOS
def kdeos(X, k):
    nbrs = NearestNeighbors(n_neighbors=k+1, p=1).fit(X)
    _, indices = nbrs.kneighbors(X)
    scores = []
    for i in range(len(X)):
        local_data = X[indices[i, 1:]]
        if local_data.shape[0] < X.shape[1]:
            scores.append(0)  # Skip ill-posed KDE estimation
            continue
        try:
            kde = gaussian_kde(local_data.T)
            score = kde(X[i].reshape(-1, 1))
            scores.append(1 / (score[0] + 1e-10))
        except np.linalg.LinAlgError:
            scores.append(0)
    return np.array(scores)

# LDF
def ldf(X, k):
    nbrs = NearestNeighbors(n_neighbors=k+1, p=1).fit(X)
    distances, _ = nbrs.kneighbors(X)
    density = 1 / (np.mean(distances[:, 1:], axis=1) + 1e-8)
    return 1 / density

# LoOP
def loop(X, k, lambda_val=3.0):
    nbrs = NearestNeighbors(n_neighbors=k+1, p=1).fit(X)
    distances, indices = nbrs.kneighbors(X)
    distances, indices = distances[:, 1:], indices[:, 1:]

    pdists = np.zeros(X.shape[0])
    for i in range(X.shape[0]):
        xi = X[i]
        neighbors = X[indices[i]]
        sigma_sq = np.mean(np.square(np.linalg.norm(neighbors - xi, axis=1, ord=1)))
        pdists[i] = np.sqrt(sigma_sq)

    loop_scores = np.zeros(X.shape[0])
    for i in range(X.shape[0]):
        neighbor_pdists = pdists[indices[i]]
        mean_sigma = np.mean(neighbor_pdists)
        plof = (pdists[i] / (mean_sigma + 1e-8)) - 1
        loop_scores[i] = norm.cdf(plof * lambda_val)
    return loop_scores

# INFLO
def inflo(X, k):
    n = X.shape[0]
    nbrs = NearestNeighbors(n_neighbors=k+1, p=1).fit(X)
    distances, indices = nbrs.kneighbors(X)
    indices = indices[:, 1:]

    reverse_knn = [[] for _ in range(n)]
    for i in range(n):
        for j in indices[i]:
            reverse_knn[j].append(i)

    densities = np.zeros(n)
    for i in range(n):
        neighbors = X[indices[i]]
        densities[i] = 1 / (np.mean(np.linalg.norm(neighbors - X[i], axis=1, ord=1)) + 1e-8)

    scores = np.zeros(n)
    for i in range(n):
        inf_neighbors = list(set(indices[i]) | set(reverse_knn[i]))
        inf_density = np.mean(densities[inf_neighbors]) if inf_neighbors else 1e-8
        scores[i] = inf_density / (densities[i] + 1e-8)
    return scores

# Evaluate AUC
def evaluate(y,scores):
    auc = roc_auc_score(y, scores)
    return auc

functions = [
    simplified_lof,
    ldof,
    odin,
    kdeos,
    ldf,
    loop,
    inflo
]


In [23]:
class ParametricMethodStateOfArt:
    def __init__(self,filename,p,logTrue=False):
        self.distance = []
        self.fileName = filename
        self.X = 0
        self.y = 0
        self.arr = []
        self.logTrue = logTrue
        self.p = p
        self.tots = []
        self.dataframe = pd.DataFrame()
        self.endValues = []

    def generateOutput(self,function):
        self._readArff()
        for a in range(1,70):
            scores = function(self.X,a)
            scores = np.nan_to_num(scores, nan=0.0)
            scores = list(scores)
            self.tots += [evaluate(self.y,scores)]
        return self._printResults(self.tots)


    def _readArff(self):
        arff_file = arff.loadarff(f'./{self.fileName}') # import the attribute-relation file format
        df4 = pd.DataFrame(arff_file[0])
        self.X = df4.drop(columns=['outlier','id']).values
        #get outlier values
        self.y = df4['outlier'].values
        le = LabelEncoder()
        #encoded the variables as 0=non-outlier, 1=outlier
        self.y = le.fit_transform(self.y)

    def _printResults(self,totalArr):
        newarr = np.nan_to_num(totalArr)
        newarr = list(newarr)
        #print(max(newarr),newarr.index(max(newarr))+2) #print the max values, the k value, and the array
        return max(newarr),newarr.index(max(newarr))+2

In [24]:
folder_structure_1d = [
    "semantic/Annthyroid/Annthyroid_withoutdupl_norm_07.arff",
    "semantic/Arrhythmia/Arrhythmia_withoutdupl_norm_46.arff",
    "semantic/Cardiotocography/Cardiotocography_withoutdupl_norm_22.arff",
    "semantic/HeartDisease/HeartDisease_withoutdupl_norm_44.arff",
    "semantic/Hepatitis/Hepatitis_withoutdupl_norm_16.arff",
    "semantic/InternetAds/InternetAds_withoutdupl_norm_19.arff",
    "semantic/PageBlocks/PageBlocks_withoutdupl_norm_09.arff",
    "semantic/Parkinson/Parkinson_withoutdupl_norm_75.arff",
    "semantic/Pima/Pima_withoutdupl_norm_35.arff",
    "semantic/SpamBase/SpamBase_withoutdupl_norm_40.arff",
    "semantic/Stamps/Stamps_withoutdupl_norm_09.arff",
    "semantic/Wilt/Wilt_withoutdupl_norm_05.arff"
]

In [25]:
endvalues = []
df = pd.DataFrame(columns=['ROC AUC', 'K'])
for n in functions:
    endvalues = []
    for z in folder_structure_1d:
        parametricMethod = ParametricMethodStateOfArt(z,p=1,logTrue=True)
        endvalues += [parametricMethod.generateOutput(n)]
    print(endvalues)
    dftest = pd.DataFrame(np.array(endvalues),columns=['ROC AUC','k'])
    df = pd.concat([df,dftest])

df.to_csv("testForStateLit.csv")

[(0.6774003117785861, 3), (0.7580773515836384, 51), (0.5377763135963998, 70), (0.6697222222222221, 70), (0.7588978185993112, 51), (0.7421173545736519, 18), (0.8475153895266891, 70), (0.7210884353741497, 6), (0.7305373134328358, 70), (0.6402688931024345, 70), (0.9104290635765738, 70), (0.5668012016028195, 3)]
[(0.7387951376170234, 28), (0.7518104408721948, 70), (0.5617017896579024, 50), (0.5432222222222222, 5), (0.7290470723306544, 69), (0.64681905370844, 41), (0.8020186079756498, 70), (0.5297619047619047, 23), (0.5850298507462687, 65), (0.5, 2), (0.7069631485541288, 69), (0.6983036998244678, 16)]
[(0.6766873099300628, 2), (0.7606040108228554, 44), (0.5576807887828659, 70), (0.6998888888888889, 69), (0.7898966704936854, 26), (0.722096574522501, 14), (0.8728066159906518, 70), (0.7373866213151927, 6), (0.7360373134328357, 68), (0.6504956282371211, 40), (0.919093851132686, 64), (0.5617766117325155, 2)]
[(0.5, 2), (0.5, 2), (0.5032188841201717, 36), (0.65425, 53), (0.7675086107921929, 36), 

In [28]:
# Assuming your list of tuples is stored in a variable called `data`
data = [
    (0.6774003117785861, 3), (0.7580773518536384, 51), (0.5377763135963998, 70), (0.6697222222222221, 70), (0.7588978185993112, 51), (0.7421173545736519, 18),
    (0.847513859256891, 70), (0.7210884353741497, 6), (0.7305373134328358, 70), (0.6402688931024345, 70), (0.9104290635756738, 70), (0.566802110628195, 3),
    (0.7891573176107234, 28), (0.7518144604879122, 6), (0.5617017896597024, 50), (0.5432222222222222, 5), (0.72904723306544, 69), (0.64681935078044, 41),
    (0.8021086079756498, 70), (0.5297619047619047, 23), (0.585029507462687, 65), (0.5, 2), (0.7069631485541288, 69), (0.698365398824468, 16),
    (0.6766873097300628, 2), (0.7606040180282554, 44), (0.557608787828659, 70), (0.6998888888888889, 69), (0.7898966794938654, 26), (0.722096754522501, 14),
    (0.8728066159906153, 70), (0.7373866213151927, 6), (0.7360373134328357, 68), (0.6504956282371211, 40), (0.9190983511332686, 64), (0.561776173251555, 2),
    (0.5, 2), (0.5, 2), (0.503218841201717, 36), (0.65425, 53), (0.70750810792192, 36), (0.5, 2), (0.649880738697281, 70), (0.7694160997732427, 57),
    (0.667865676146792, 70), (0.5, 2), (0.785102050428542, 70), (0.709467654692494, 62),
    (0.6774003117785861, 3), (0.7580773518536384, 51), (0.5377763135963998, 70), (0.6697222222222221, 70), (0.7588978185993112, 51), (0.7421173545736519, 18),
    (0.847513859256891, 70), (0.7210884353741497, 6), (0.7305373134328358, 70), (0.6402688931024345, 70), (0.9104290635756738, 70), (0.566802110628195, 3),
    (0.7209470914343199, 38), (0.7576049973515127, 70), (0.5683596191507979, 21), (0.5555, 70), (0.7416762342135477, 65), (0.6527656835675791, 70), (0.7742656595712214, 70), (0.5756082721088435, 19), (0.6153419521337343, 69), (0.47210689432375455, 3), (0.7732228365871246, 70), (0.6835272236087888, 14),
    (0.7131290713087033, 31), (0.752984263159255, 70), (0.5797845743572649, 69), (0.5631666666666668, 68), (0.746286567614618, 64), (0.680269178864885, 70),
    (0.75350314319548, 70), (0.5250850340136055, 11), (0.6162089552238805, 70), (0.5069373110501278, 2), (0.7368201273619375, 70), (0.7020557233925322, 6)
]


# Convert to CSV string format
csv_string = "Score,k\n" + "\n".join([f"{score},{k}" for score, k in data])

# Save to a CSV file (optional)
with open("outlier_scores.csv", "w") as f:
    f.write(csv_string)

# Or just print if you want to copy-paste to Google Sheets manually
print(csv_string)


Score,k
0.6774003117785861,3
0.7580773518536384,51
0.5377763135963998,70
0.6697222222222221,70
0.7588978185993112,51
0.7421173545736519,18
0.847513859256891,70
0.7210884353741497,6
0.7305373134328358,70
0.6402688931024345,70
0.9104290635756738,70
0.566802110628195,3
0.7891573176107234,28
0.7518144604879122,6
0.5617017896597024,50
0.5432222222222222,5
0.72904723306544,69
0.64681935078044,41
0.8021086079756498,70
0.5297619047619047,23
0.585029507462687,65
0.5,2
0.7069631485541288,69
0.698365398824468,16
0.6766873097300627,2
0.7606040180282554,44
0.557608787828659,70
0.6998888888888889,69
0.7898966794938654,26
0.722096754522501,14
0.8728066159906153,70
0.7373866213151927,6
0.7360373134328357,68
0.6504956282371211,40
0.9190983511332687,64
0.561776173251555,2
0.5,2
0.5,2
0.503218841201717,36
0.65425,53
0.70750810792192,36
0.5,2
0.649880738697281,70
0.7694160997732427,57
0.667865676146792,70
0.5,2
0.785102050428542,70
0.709467654692494,62
0.6774003117785861,3
0.7580773518536384,51
0.53777631

In [30]:
literature_dataset_paths = [
    "literature/ALOI/ALOI_withoutdupl_norm.arff",
    "literature/Glass/Glass_withoutdupl_norm.arff",
    "literature/Ionosphere/Ionosphere_withoutdupl_norm.arff",
    "literature/KDDCup99/KDDCup99_withoutdupl_norm_idf.arff",
    "literature/Lymphography/Lymphography_withoutdupl_norm_idf.arff",
    "literature/PenDigits/PenDigits_withoutdupl_norm_v10.arff",
    "literature/Shuttle/Shuttle_withoutdupl_norm_v10.arff",
    "literature/Waveform/Waveform_withoutdupl_norm_v10.arff",
    "literature/WBC/WBC_withoutdupl_norm_v10.arff",
    "literature/WDBC/WDBC_withoutdupl_norm_v10.arff",
    "literature/WPBC/WPBC_withoutdupl_norm.arff"
]



In [31]:
endvalues = []
df = pd.DataFrame(columns=['ROC AUC', 'K'])
for n in functions:
    endvalues = []
    for z in literature_dataset_paths:
        parametricMethod = ParametricMethodStateOfArt(z,p=1,logTrue=True)
        endvalues += [parametricMethod.generateOutput(n)]
    print(endvalues)
    dftest = pd.DataFrame(np.array(endvalues),columns=['ROC AUC','k'])
    df = pd.concat([df,dftest])

df.to_csv("testForStateLit.csv")

[(0.7486094443648506, 3), (0.8799457994579946, 2), (0.9003527336860669, 2), (0.9539617640306388, 70), (1.0, 3), (0.9913383428107231, 21), (0.7799999999999999, 14), (0.7777355668561173, 70), (0.9971830985915494, 22), (0.9871148459383753, 57), (0.5269832323516979, 29)]
[(0.7524460253127698, 9), (0.781029810298103, 26), (0.8322045855379189, 50), (0.7708676142174358, 70), (0.9964788732394366, 44), (0.7291582047116165, 70), (0.7797692307692308, 22), (0.6959288064612623, 67), (0.9436619718309859, 70), (0.9795518207282914, 70), (0.5017613075947583, 61)]
[(0.7461519379257544, 3), (0.8799457994579946, 2), (0.9003527336860669, 2), (0.9701241312378687, 70), (1.0, 8), (0.9911581031681559, 12), (0.8467692307692308, 5), (0.7857014657493269, 66), (0.9974178403755869, 25), (0.9896358543417367, 70), (0.530999013667747, 19)]
[(0.5225707068927408, 62), (0.8395663956639566, 19), (0.8624514991181658, 70), (0.5, 2), (0.8274647887323944, 33), (0.8669146019496344, 59), (0.7730384615384616, 48), (0.65141788812